In [1]:
import torch
import torch.nn as nn
import torchvision.models as models

class DeepFakeEfficientNet(nn.Module):
    def __init__(self, pretrained=True):
        super(DeepFakeEfficientNet, self).__init__()
        
        # 1. Load the pre-trained EfficientNet-B4 base
        # B4 expects an input resolution of 380x380 pixels
        if pretrained:
            weights = models.EfficientNet_B4_Weights.DEFAULT
        else:
            weights = None
            
        self.base_model = models.efficientnet_b4(weights=weights)
        
        # 2. Extract the number of input features going into the final layer
        # For EfficientNet-B4, the classifier's in_features is 1792
        in_features = self.base_model.classifier[1].in_features
        
        # 3. Replace the final classification head for Binary Classification (Real vs Fake)
        self.base_model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True), # Helps prevent overfitting on fake textures
            nn.Linear(in_features, 1)        # Output 1 logit for Binary Cross Entropy Loss
        )
        
    def forward(self, x):
        return self.base_model(x)

# Initialize the model and move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepFakeEfficientNet(pretrained=True).to(device)

print(f"Model initialized successfully and loaded onto: {device}")

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 192MB/s] 


Model initialized successfully and loaded onto: cuda


In [11]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 1. Define Training & Validation Transformations
train_transforms = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.RandomRotation(degrees=15),   
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Custom Dataset Loader with Transparency Fix
class DeepFakeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path)
        
        # Fixing palette/transparency images before applying transforms
        if image.mode in ('RGBA', 'LA') or (image.mode == 'P' and 'transparency' in image.info):
            image = image.convert("RGBA").convert("RGB")
        else:
            image = image.convert("RGB")
            
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

print("Data transformations and Dataset class are ready without any error!")

Data transformations and Dataset class are ready without any error!


In [12]:
import os
from sklearn.model_selection import train_test_split

# 1. Point to your Kaggle dataset root path
# Adjust the directory name if your specific dataset path looks slightly different
dataset_dir = '/kaggle/input/datasets/saurabhbagchi/deepfake-image-detection'

image_paths = []
labels = []

# 2. Automatically crawl the directory for Real (0) and Fake (1) images
for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(root, file)
            
            # Label based on folder naming conventions
            if 'real' in root.lower():
                image_paths.append(full_path)
                labels.append(0)  # Real = 0
            elif 'fake' in root.lower() or 'synthesized' in root.lower():
                image_paths.append(full_path)
                labels.append(1)  # Fake = 1

print(f"Total images found: {len(image_paths)}")
print(f"Real images: {labels.count(0)} | Fake images: {labels.count(1)}")

# 3. Create your Train / Validation Splits (80/20)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

# 4. Instantiate the Custom Dataset classes we wrote earlier
train_dataset = DeepFakeDataset(train_paths, train_labels, transform=train_transforms)
val_dataset = DeepFakeDataset(val_paths, val_labels, transform=val_transforms)

# 5. Build PyTorch DataLoaders
# Batch size 16/32 works great with EfficientNet-B4 on Kaggle's T4 GPU
BATCH_SIZE = 16 

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders created cleanly! Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")

Total images found: 983
Real images: 436 | Fake images: 547
DataLoaders created cleanly! Train Batches: 50 | Val Batches: 13


In [13]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 1. Loss Function and Optimizer Setup
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# Scheduler to lower LR if validation loss plateaus
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# 2. Training Configurations
EPOCHS = 10
best_val_loss = float('inf')

print("Starting training loop for EfficientNet-B4...")

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
        # Calculate accuracy
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        
    epoch_train_loss = train_loss / total_train
    epoch_train_acc = correct_train / total_train

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val / total_val
    
    # Step the scheduler based on validation loss
    scheduler.step(epoch_val_loss)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] -> "
          f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")
    
    # Save the best model weights
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), 'best_deepfake_b4_model.pth')
        print("=> Saved new best model checkpoint!")

Starting training loop for EfficientNet-B4...
Epoch [1/10] -> Train Loss: 0.6270, Train Acc: 0.6527 | Val Loss: 0.6064, Val Acc: 0.6701
=> Saved new best model checkpoint!
Epoch [2/10] -> Train Loss: 0.5575, Train Acc: 0.7328 | Val Loss: 0.5446, Val Acc: 0.7868
=> Saved new best model checkpoint!
Epoch [3/10] -> Train Loss: 0.4750, Train Acc: 0.8193 | Val Loss: 0.4739, Val Acc: 0.8223
=> Saved new best model checkpoint!
Epoch [4/10] -> Train Loss: 0.4031, Train Acc: 0.8486 | Val Loss: 0.4293, Val Acc: 0.8477
=> Saved new best model checkpoint!
Epoch [5/10] -> Train Loss: 0.3145, Train Acc: 0.8893 | Val Loss: 0.4002, Val Acc: 0.8426
=> Saved new best model checkpoint!
Epoch [6/10] -> Train Loss: 0.2521, Train Acc: 0.9211 | Val Loss: 0.3847, Val Acc: 0.8477
=> Saved new best model checkpoint!
Epoch [7/10] -> Train Loss: 0.2147, Train Acc: 0.9224 | Val Loss: 0.4024, Val Acc: 0.8579
Epoch [8/10] -> Train Loss: 0.1541, Train Acc: 0.9567 | Val Loss: 0.4221, Val Acc: 0.8731
Epoch [9/10] -> Tr

In [14]:
import torch
from torchvision import transforms
from PIL import Image
import torch.nn.functional as F

def predict_deepfake(image_path, model, device):
    # 1. Same exact preprocessing pipeline used during validation
    test_transforms = transforms.Compose([
        transforms.Resize((380, 380)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # 2. Load and fix transparency if any
    image = Image.open(image_path)
    if image.mode in ('RGBA', 'LA') or (image.mode == 'P' and 'transparency' in image.info):
        image = image.convert("RGBA").convert("RGB")
    else:
        image = image.convert("RGB")
        
    # 3. Apply transforms and add batch dimension
    input_tensor = test_transforms(image).unsqueeze(0).to(device)
    
    # 4. Inference mode
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        # Apply sigmoid to get probability between 0 and 1
        probability = torch.sigmoid(output).item()
        
    # 5. Format results (Recall: Real = 0, Fake = 1)
    if probability > 0.5:
        confidence = probability * 100
        print(f"Prediction: 🚨 FAKE IMAGE (Confidence: {confidence:.2f}%)")
    else:
        confidence = (1 - probability) * 100
        print(f"Prediction: ✅ REAL IMAGE (Confidence: {confidence:.2f}%)")

# ==========================================
# TEST CORNER: Apni kisi bhi image ka path yahan daaliye
# ==========================================
# Example: Aap apne val_paths se koi random image utha kar check kar sakte hain
sample_image_path = val_paths[0] 

print(f"Testing on image: {sample_image_path}")
predict_deepfake(sample_image_path, model, device)

Testing on image: /kaggle/input/datasets/saurabhbagchi/deepfake-image-detection/test-20250112T065939Z-001/test/fake/95.jpg
Prediction: 🚨 FAKE IMAGE (Confidence: 99.92%)


In [15]:
# Final step to ensure your best weights are safely stored
import shutil

# Humne training loop mein already 'best_deepfake_b4_model.pth' save kiya tha.
# Chalo verify karte hain aur use safe location par confirm karte hain.
if os.path.exists('best_deepfake_b4_model.pth'):
    print("✓ Your best model weights are safely saved as 'best_deepfake_b4_model.pth'!")
    print("You can see it in the right sidebar under the 'Output' folder.")
else:
    # Safe guard: Agar training loop se check missed hua ho, toh current state save kar lo
    torch.save(model.state_dict(), 'final_deepfake_b4_model.pth')
    print("✓ Model successfully saved as 'final_deepfake_b4_model.pth'!")
    

✓ Your best model weights are safely saved as 'best_deepfake_b4_model.pth'!
You can see it in the right sidebar under the 'Output' folder.
